In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import os

DATA_FOLDER = r"D:\cowork\Project Mortgage Deliquency\Project Data set"

COLS_TO_USE = [1, 2, 7, 9, 19, 20, 22, 23, 30, 34, 39, 43]

COL_NAMES = [
    "loan_id", "reporting_period", "orig_interest_rate", "orig_upb",
    "orig_ltv", "orig_cltv", "dti", "credit_score",
    "property_state", "amortization_type", "delinquency_status", "zero_balance_code"
]

print("All imports loaded successfully")

In [ ]:
csv_path = r"D:\cowork\Project Mortgage Deliquency\Project Data set\2021Q1.csv"

Final_db = []
for chunk in pd.read_csv(csv_path, sep="|", header=None, dtype=str, usecols=COLS_TO_USE, chunksize=300_000):

    chunk.columns = COL_NAMES

    chunk["delinquency_num"] = pd.to_numeric(
        chunk["delinquency_status"], errors="coerce"
    ).fillna(0).astype(int)

    aggregate_db = chunk.groupby("loan_id").agg(
        orig_interest_rate=("orig_interest_rate", "first"),
        orig_upb=("orig_upb", "first"),
        orig_ltv=("orig_ltv", "first"),
        orig_cltv=("orig_cltv", "first"),
        dti=("dti", "first"),
        credit_score=("credit_score", "first"),
        property_state=("property_state", "first"),
        amortization_type=("amortization_type", "first"),
        worst_delinquency=("delinquency_num", "max")
    ).reset_index()

    Final_db.append(aggregate_db)

df_loans = pd.concat(Final_db, ignore_index=True)
df_loans = df_loans.groupby("loan_id").agg(
    orig_interest_rate=("orig_interest_rate", "first"),
    orig_upb=("orig_upb", "first"),
    orig_ltv=("orig_ltv", "first"),
    orig_cltv=("orig_cltv", "first"),
    dti=("dti", "first"),
    credit_score=("credit_score", "first"),
    property_state=("property_state", "first"),
    amortization_type=("amortization_type", "first"),
    worst_delinquency=("worst_delinquency", "max")
).reset_index()

print(f"Done, unique loans: {len(df_loans):,}")
print(df_loans.head())

In [ ]:
print(df_loans.tail(10))

In [ ]:
df_loans.to_csv(r"D:\cowork\Project Mortgage Deliquency\df_loans_2021Q1.csv", index=False)
print("Saved")

In [ ]:
df_loans["defaulted"] = (df_loans["worst_delinquency"] >= 3).astype(int)

print(df_loans["defaulted"].value_counts())
print(f"Default rate: {df_loans['defaulted'].mean()*100:.2f}%")

In [ ]:
model_variables = ["orig_interest_rate", "orig_ltv", "dti", "credit_score"]

for col in model_variables:
    df_loans[col] = pd.to_numeric(df_loans[col], errors="coerce")

print("Total Missing values per column:")
print(df_loans[model_variables].isna().sum())

df_clean = df_loans[model_variables + ["defaulted"]].dropna()

print(f"Rows before cleaning: {len(df_loans):,}")
print(f"Rows after cleaning: {len(df_clean):,}")
print(f"Rows dropped: {len(df_loans) - len(df_clean):,}")

In [ ]:
from sklearn.model_selection import train_test_split

X = df_clean[model_variables]
Y = df_clean["defaulted"]

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

print(f"Training rows:  {len(X_train):,}")
print(f"Testing rows:   {len(X_test):,}")
print(f"Default rate in train: {Y_train.mean()*100:.2f}%")
print(f"Default rate in test:  {Y_test.mean()*100:.2f}%")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(class_weight="balanced", random_state=42, max_iter=1000)
model.fit(X_train_scaled, Y_train)

print("Model Trained Successfully")

In [ ]:
weights = pd.Series(model.coef_[0], index=model_variables)
print("Model Weights (Risk Impact):\n", weights.sort_values(ascending=False))

In [ ]:
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
                             confusion_matrix, classification_report)

Y_pred_prob = model.predict_proba(X_test_scaled)[:, 1]
Y_pred = model.predict(X_test_scaled)

auc_roc = roc_auc_score(Y_test, Y_pred_prob)
auc_pr = average_precision_score(Y_test, Y_pred_prob)

print(f"AUC-ROC:              {auc_roc:.4f}")
print(f"Precision-Recall AUC: {auc_pr:.4f}")
print()

cm = confusion_matrix(Y_test, Y_pred)
print("Confusion matrix\n\n", cm)
print("\nTrue Negatives  (TN) =", cm[0, 0])
print("False Positives (FP) =", cm[0, 1])
print("False Negatives (FN) =", cm[1, 0])
print("True Positives  (TP) =", cm[1, 1])
print()
print(classification_report(Y_test, Y_pred))

In [ ]:
for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    Y_pred_custom = (Y_pred_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(Y_test, Y_pred_custom).ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    print(f"Threshold {threshold:.1f} -> Caught: {tp:,} defaults | False alarms: {fp:,} | Precision: {precision:.2f} | Recall: {recall:.2f}")

In [ ]:
coef_df = pd.DataFrame({
    "feature": model_variables,
    "coefficient": model.coef_[0]
}).sort_values("coefficient")

colors = ["red" if c > 0 else "green" for c in coef_df["coefficient"]]

plt.figure(figsize=(8, 5))
plt.barh(coef_df["feature"], coef_df["coefficient"], color=colors)
plt.axvline(x=0, color="black", linewidth=0.8)
plt.title("Feature Coefficients - Logistic Regression\n(Red = increases default risk, Green = decreases default risk)")
plt.xlabel("Coefficient value")
plt.tight_layout()
plt.show()